# Nettoyage et Pré-traitement des données (Preprocessing)

Ce notebook permet d'appliquer les étapes de nettoyage, d'encodage catégoriel, de découpage train/test et de standardisation des variables numériques. Les fonctions de logique métier sont importées depuis le module `preprocessing.py` afin de respecter la séparation des responsabilités.

In [9]:
import sys
import os

# Ajout du chemin racine du projet pour résoudre les imports de lead_scoring
sys.path.append(os.path.abspath(os.path.join("..", "..")))

import pandas as pd
import numpy as np

# Import des modules du projet
from lead_scoring.src.data_loader import charger_donnees
from lead_scoring.src.preprocessing import nettoyer_donnees, encoder_variables_cat, separer_train_test, normaliser_donnees

## 1. Chargement des données

On charge les données brutes via le loader du projet, qui applique automatiquement l'encodage initial de la cible `deposit` (yes -> 1, no -> 0).

In [ ]:
df_brut = charger_donnees()
print(f"Dimensions brutes : {df_brut.shape}")
df_brut.head()

## 2. Nettoyage des données

On applique `nettoyer_donnees()` pour :
- Vérifier la présence de valeurs manquantes (sanity check).
- Traiter la variable `pdays = -1` (création de l'indicateur binaire et remplacement par 0).
- Encoder les variables binaires `default`, `housing`, `loan` (yes/no -> 1/0).

In [10]:
df_nettoye = nettoyer_donnees(df_brut)

print("\nAperçu des colonnes nettoyées (pdays, pdays_jamais_contacte, default, housing, loan) :")
colonnes_verif = ["pdays", "pdays_jamais_contacte", "default", "housing", "loan"]
df_nettoye[colonnes_verif].head(10)

[Sanity Check] Aucune valeur manquante détectée.
Traitement de 'pdays = -1' effectué : création de la variable binaire 'pdays_jamais_contacte'.
Encodage de la colonne binaire 'default' effectué (yes -> 1, no -> 0).
Encodage de la colonne binaire 'housing' effectué (yes -> 1, no -> 0).
Encodage de la colonne binaire 'loan' effectué (yes -> 1, no -> 0).

Aperçu des colonnes nettoyées (pdays, pdays_jamais_contacte, default, housing, loan) :


,pdays,pdays_jamais_contacte,default,housing,loan
0,0,1,0,1,0
1,0,1,0,0,0
2,0,1,0,1,0
3,0,1,0,1,0
4,0,1,0,0,0
5,0,1,0,1,1
6,0,1,0,1,1
7,0,1,0,1,0
8,0,1,0,1,0
9,0,1,0,1,0


## 3. Encodage des variables catégorielles

On applique le One-Hot Encoding sur les variables catégorielles multi-classes pour obtenir un dataset entièrement numérique.

In [11]:
df_encode = encoder_variables_cat(df_nettoye)
print(f"Nombre de colonnes initial : {df_nettoye.shape[1]}")
print(f"Nombre de colonnes après One-Hot Encoding : {df_encode.shape[1]}")
df_encode.head()

Encodage catégoriel effectué sur les colonnes : ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
Nombre de colonnes initial : 18
Nombre de colonnes après One-Hot Encoding : 44


,age,default,balance,housing,loan,day,duration,campaign,pdays,previous,...,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown
0,59,0,2343,1,0,5,1042,1,0,0,...,0,0,0,1,0,0,0,0,0,1
1,56,0,45,0,0,5,1467,1,0,0,...,0,0,0,1,0,0,0,0,0,1
2,41,0,1270,1,0,5,1389,1,0,0,...,0,0,0,1,0,0,0,0,0,1
3,55,0,2476,1,0,5,579,1,0,0,...,0,0,0,1,0,0,0,0,0,1
4,54,0,184,0,0,5,673,2,0,0,...,0,0,0,1,0,0,0,0,0,1


## 4. Séparation Train/Test

Découpage des données en ensemble d'entraînement (80%) et de test (20%) avec une graine fixe de reproductibilité (`random_state=42`).

In [12]:
X_train, X_test, y_train, y_test = separer_train_test(df_encode, cible="deposit", test_size=0.2, random_state=42)

Découpage Train/Test effectué (ratio 80%/20%).
Dimensions - X_train : (8929, 43), X_test : (2233, 43)


## 5. Normalisation des variables numériques

Mise à l'échelle via `StandardScaler` appliquée sur les colonnes numériques pour que chacune ait une moyenne nulle et un écart-type égal à 1. 

**Rappel :** L'ajustement du scaler s'effectue uniquement sur `X_train` pour éviter toute fuite d'information (data leakage).

In [13]:
colonnes_numeriques = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
X_train_scaled, X_test_scaled, scaler = normaliser_donnees(X_train, X_test, colonnes_numeriques)

print("\nVérification des statistiques de X_train_scaled (Moyenne ~0, Écart-type ~1) :")
print(X_train_scaled[colonnes_numeriques].agg(["mean", "std"]).round(4))

Normalisation StandardScaler effectuée sur les colonnes : ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Vérification des statistiques de X_train_scaled (Moyenne ~0, Écart-type ~1) :
         age  balance     day  duration  campaign   pdays  previous
mean  0.0000  -0.0000  0.0000    0.0000   -0.0000  0.0000   -0.0000
std   1.0001   1.0001  1.0001    1.0001    1.0001  1.0001    1.0001


## 6. Sauvegarde des datasets préparés

Sauvegarde dans `lead_scoring/data/processed/` sous forme de fichiers CSV pour pouvoir être réutilisés en Semaine 3 pour la modélisation.

In [14]:
dossier_sortie = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "processed"))
os.makedirs(dossier_sortie, exist_ok=True)

X_train_scaled.to_csv(os.path.join(dossier_sortie, "X_train.csv"), index=False)
X_test_scaled.to_csv(os.path.join(dossier_sortie, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(dossier_sortie, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(dossier_sortie, "y_test.csv"), index=False)

print(f"Fichiers enregistrés avec succès dans : {dossier_sortie}")

Fichiers enregistrés avec succès dans : c:\Users\Yohan\UQAC\python_UQAC\projet_optimisateur_marketing\lead_scoring\data\processed
